# Forecasting with a Temporal Fusion Transformer

Notebook 01 filled in the demand stockouts hid. This notebook forecasts it.

| Step | What it does |
|---|---|
| 1 | load the recovered demand |
| 2 | search hyperparameters, then train the winner |
| 3 | compare against the baselines |
| 4 | train the same model on raw sales - does recovery help? |
| 5-7 | where recovery helps, and what the trade costs |

A **range**, not a number: the model outputs six quantiles, because ordering perishables means knowing
the downside as well as the middle. The test week is never touched here.

Notebook 03 runs the same comparison with a gradient-boosted forecaster. That one is deterministic and
fits in about a minute, and it scores level with this one - so the recovery result does not depend on
the transformer.

Runs on CPU, but a GPU is far faster (Colab: *Runtime -> Change runtime type -> GPU*).

## Setup

In [1]:
import sys
sys.path.insert(0, "..")   # notebooks/ is one level down, so the repo root has to go on the path

In [2]:
import warnings

import pandas as pd
import torch
warnings.filterwarnings("ignore")   # pytorch-forecasting is noisy about dataloader workers

from src.utils import config, data_io
from src.utils.metrics import quantile_scores
from src import forecast, recovery

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    gpu = "Apple GPU (MPS)"
else:
    gpu = "none (CPU)"
print(f"torch {torch.__version__} | GPU: {gpu}")

torch 2.13.0+cpu | GPU: none (CPU)


## 1. Load the recovered demand

In [3]:
# The recovered parquets are a superset of the raw daily ones and are the pair that is
# committed, so this one read works from a fresh clone. Merging the raw daily file in
# would need a rebuild that Colab has not done.
daily = recovery.load_daily("recovered")

assert daily["recovered_demand"].notna().all(), "run notebook 01 step 4 first"
print(f"{len(daily):,} rows | recovery model:", recovery.load_params()["model"])
daily.groupby("period")[["sale_amount", "recovered_demand"]].mean().round(4)

loading recovered subset from data/processed
543,297 rows | recovery model: lightgbm_tweedie


,sale_amount,recovered_demand
period,,
calibration,1.1303,1.2861
test,1.1990,1.3506
training,0.9555,1.1457
validation,1.0471,1.1857


## 2. Hyperparameter search

`forecast.GRID` defines the configurations searched. Every setting in it is fitted and scored on the
**validation** window; anything not in it is held at its default in `forecast.train`.

| setting | what it controls |
|---|---|
| `learning_rate` | step size. Too high and it never settles; too low and it stops before learning |
| `encoder_days` | history read per forecast. Longer sees more seasonality but gives fewer training windows |
| `hidden_size` | model capacity. More data supports more of it |
| `dropout` | regularisation strength |

An earlier full-factorial search on a smaller subset found that **almost nothing moved validation
pinball**. That result is why this grid is small: settings that made no measurable difference are held
fixed at the level that was best averaged over all the others, and only the settings with a specific
reason to be re-tested on this larger subset are searched.

**Scored on `pinball(avg)`**, not WAPE: the ordering stage consumes the whole q10/q50/q90 range, so
the metric that scores the range should pick the model.

In [4]:
TUNE = False   # False = load the saved ranking; re-run after a disconnect to resume

tuning = (forecast.tune(daily, max_epochs=15) if TUNE
          else pd.read_csv(config.tft_tuning("recovered")))
tuning.drop(columns="config_id").round(4)

,learning_rate,dropout,encoder_days,weight_decay,WAPE,WPE,MAE,pinball(avg),CRPS~
0,0.010,0.3,3,0.0001,0.3151,0.0405,0.3220,0.1031,0.2006
1,0.010,0.3,7,0.0001,0.3212,0.0788,0.3290,0.1055,0.2054
2,0.005,0.3,3,0.0001,0.3212,0.0835,0.3289,0.1055,0.2054
3,0.010,0.2,7,0.0001,0.3237,0.0920,0.3312,0.1063,0.2068
4,0.010,0.2,5,0.0001,0.3258,0.0927,0.3338,0.1063,0.2069
5,0.010,0.2,3,0.0001,0.3289,0.0743,0.3367,0.1071,0.2085
6,0.005,0.3,7,0.0001,0.3253,0.0991,0.3341,0.1072,0.2087
7,0.005,0.2,3,0.0001,0.3296,0.1079,0.3384,0.1072,0.2086
8,0.005,0.3,5,0.0001,0.3279,0.0905,0.3365,0.1077,0.2097
9,0.005,0.2,5,0.0001,0.3318,0.1224,0.3402,0.1099,0.2140


### Train the best settings

The winner is refitted with the same epoch ceiling as the search, so the model that is saved is the
one the ranking actually measured.

| | |
|---|---|
| **Target** | `recovered_demand` (raw `sale_amount` in step 4) |
| **History** | the winning `encoder_days`, read by the model itself - no hand-built lag columns |
| **Known ahead** | day of week, discount, holiday, activity, weather |
| **Per product** | store, product and the three category IDs, as learned embeddings |
| **Horizon** | 7 days, rolled forward across the period |

Early stopping watches `val_loss` and the best epoch is checkpointed and reloaded, so the saved model
is the best one seen rather than the last one trained. Each 7-day block is forecast from history that
stops the day before it starts, so no block sees inside itself.

In [5]:
BEST = forecast.best_params(tuning)
print("winning config:", BEST)

TRAIN   = False
PERIODS = ("validation", "calibration")

if TRAIN:
    recovered = forecast.run(daily, tag="recovered", max_epochs=15, **BEST)
else:
    recovered = {p: pd.read_parquet(config.forecast_parquet(p, "recovered")) for p in PERIODS}

recovered["validation"][["dt", "store_id", "product_id",
                         "q10", "q50", "q90", "sale_amount", "is_censored"]].head()

winning config: {'learning_rate': 0.01, 'dropout': 0.3, 'encoder_days': 3, 'weight_decay': 0.0001}


,dt,store_id,product_id,q10,q50,q90,sale_amount,is_censored
0,2024-05-29,10,116,0.326862,0.601307,0.945230,0.9,1
1,2024-05-30,10,116,0.289122,0.550583,0.876019,0.9,0
2,2024-05-31,10,116,0.328808,0.604475,0.943175,0.4,0
3,2024-06-01,10,116,0.438102,0.756338,1.149172,0.6,0
4,2024-06-02,10,116,0.488201,0.824321,1.235867,0.2,0


## 3. Compare against the baselines

Same scoring as notebook 01: **per date, on non-stockout rows only**, against recorded `sale_amount`,
so these numbers stay comparable to the baseline scorecard.

- **WAPE / MAE** - how far off the middle guess (`q50`) is. Lower is better.
- **WPE** - direction. Positive over-forecasts, negative under-forecasts.
- **pinball(avg) / CRPS~** - whether the whole range is right, not just the middle. This is what the
  ordering stage consumes, so it decides the model.

> **The gaps here are small.** This model and the gradient-boosted forecaster in notebook 03 land
> within about 1% of each other on pinball, which is the point rather than a disappointment: the
> recovery result holds under two unrelated model families.

In [6]:
baselines = pd.read_csv(config.BASELINE_SCORECARD, index_col=0)

pd.concat([pd.DataFrame({"tft_recovered": quantile_scores(recovered["validation"])}).T,
           baselines]).round(4)

,WAPE,WPE,MAE,pinball(avg),CRPS~,pinball@0.5,n_series,n_failed
tft_recovered,0.3284,0.0851,0.3358,0.1074,0.2091,NaN,NaN,NaN
seasonal_naive,0.4213,0.0187,0.4301,NaN,NaN,0.2212,NaN,NaN
xgboost_quantile,0.3423,-0.0187,0.3502,0.1121,0.2181,NaN,NaN,NaN
sarima,0.5210,-0.2521,0.4149,NaN,NaN,NaN,30.0,0.0


## 4. Recovered demand vs raw sales

The same settings, seed and features, trained on raw `sale_amount` instead of `recovered_demand`.
Only the target changes.

**Pooled, this table cannot settle which is better** - and it will make recovery look slightly
worse. Scoring skips stockout rows, and stockout rows are the only rows recovery changes, so on the
rows being scored the two targets are *identical*. The raw version predicts exactly what it is
measured against; the recovered version predicts demand and is marked down for exceeding recorded
sales.

The question that *can* be settled is asked in **notebook 01 section 6**, which runs this same
comparison on the XGBoost baseline and splits the score by how often each product sells out. There,
recovery removes the under-forecast on the products that sell out most, at the cost of a mild
over-forecast on everything else. Expect the TFT to behave the same way; the pooled row here is the
cost side of that trade, not a verdict.

In [7]:
# the SAME winning settings: only the target changes
if TRAIN:
    raw = forecast.run(daily, tag="raw", max_epochs=15, **BEST)
else:
    raw = {p: pd.read_parquet(config.forecast_parquet(p, "raw")) for p in PERIODS}

both = {"tft_recovered": recovered["validation"], "tft_raw": raw["validation"]}
scorecard = pd.concat([pd.DataFrame({k: quantile_scores(v) for k, v in both.items()}).T, baselines])
scorecard.to_csv(config.FORECAST_SCORECARD)
scorecard.round(4)

,WAPE,WPE,MAE,pinball(avg),CRPS~,pinball@0.5,n_series,n_failed
tft_recovered,0.3284,0.0851,0.3358,0.1074,0.2091,NaN,NaN,NaN
tft_raw,0.3286,-0.0656,0.3364,0.1089,0.2120,NaN,NaN,NaN
seasonal_naive,0.4213,0.0187,0.4301,NaN,NaN,0.2212,NaN,NaN
xgboost_quantile,0.3423,-0.0187,0.3502,0.1121,0.2181,NaN,NaN,NaN
sarima,0.5210,-0.2521,0.4149,NaN,NaN,NaN,30.0,0.0


### Forecast gap on stockout days

The scoring above hides the difference; the forecasts themselves show it. On days that ran out, the
recovered version should predict noticeably more than the raw one - and on days that did not, the two
should almost agree. That gap is the demand the raw model never learned existed.

In [8]:
keys = ["store_id", "product_id", "dt"]
gap = (recovered["validation"][keys + ["is_censored", "q50"]]
       .rename(columns={"q50": "q50_recovered"})
       .merge(raw["validation"][keys + ["q50"]].rename(columns={"q50": "q50_raw"}), on=keys))

(gap.groupby("is_censored")[["q50_raw", "q50_recovered"]].mean()
    .assign(gap_pct=lambda d: (d.q50_recovered / d.q50_raw - 1) * 100).round(4))

,q50_raw,q50_recovered,gap_pct
is_censored,,,
0,0.9618,1.1151,15.937500
1,1.0236,1.2180,18.992901


---

# Does training on recovered demand actually help?

Section 3 shows the two targets scoring almost the same. That is the question this half of the
notebook answers, and it takes three steps because the obvious comparison cannot settle it.

| step | question | why it is needed |
|---|---|---|
| **5** | Why does the pooled comparison show nothing? | It grades the wrong days. Establishes that a null here is not a finding |
| **6** | Then where does recovery help? | Splits products by how often they sell out - the axis the effect lives on |
| **7** | What is the trade in shop terms? | Lost sales against waste, because WAPE charges both the same and a shop does not |

Read in order, they answer one question. Read separately, they look like four unrelated tables.

## 5. Why the pooled scorecard cannot see recovery

Section 4 shows the two targets nearly level, and that is not a null result - it is the metric
declining to answer. Accuracy is scored on **full-shelf days only**, because those are the only days
where recorded sales are the true demand.

But full-shelf days are not a fair sample of days. **A shelf stays full on a quiet day and empties on
a busy one.** So grading only on full-shelf days means grading on the quiet ones, and a model that
learned the demand level across all days will read high there through no fault of its own.

The cell below measures how high. That figure is the handicap any demand-trained model carries into
section 3 before it makes a single mistake.

In [9]:
from src.utils.features import censoring_bucket
from src.utils.metrics import attach_bucket, lost_sales_vs_waste, scores_by_bucket

band = censoring_bucket(daily)   # each series' TRAINING-period stockout rate, cut into four groups

tr = daily[daily.period == "training"]
full_shelf, sold_out = tr[tr.is_censored == 0], tr[tr.is_censored == 1]

print("recovery changes nothing on full-shelf days: biggest difference between recovered and"
      f" recorded = {(full_shelf.recovered_demand - full_shelf.sale_amount).abs().max():.1e}")
print()
print(f"  average demand, full-shelf days : {full_shelf.recovered_demand.mean():.3f}   <- the only days we grade")
print(f"  average demand, sold-out days   : {sold_out.recovered_demand.mean():.3f}")
print(f"  average demand, all days        : {tr.recovered_demand.mean():.3f}   <- the level a model learns")
print()
reads_high = (tr.recovered_demand.mean() / full_shelf.recovered_demand.mean() - 1) * 100
print("  => a model that learns the all-day level, graded only on full-shelf days,")
print(f"     looks {reads_high:.0f}% too high before it makes a single mistake")

# attach_bucket, not a bare reindex: it casts the ID dtypes and RAISES if nothing matched, which is
# the failure mode that once produced a clean-looking table with every band NaN.
tr = tr.assign(group=attach_bucket(tr, band))
rows = []
for g, sub in tr.groupby("group", observed=True):
    fs = sub[sub.is_censored == 0]
    rows.append({"sells_out": str(g),
                 "full_shelf_days": len(fs),
                 "sold_out_days": int((sub.is_censored == 1).sum()),
                 "demand_full_shelf": round(fs.recovered_demand.mean(), 3),
                 "demand_all_days": round(sub.recovered_demand.mean(), 3),
                 "looks_high_by_%": round((sub.recovered_demand.mean()
                                           / fs.recovered_demand.mean() - 1) * 100, 1)})
print()
print(pd.DataFrame(rows).to_string(index=False))

recovery changes nothing on full-shelf days: biggest difference between recovered and recorded = 3.6e-15

  average demand, full-shelf days : 0.922   <- the only days we grade
  average demand, sold-out days   : 1.416
  average demand, all days        : 1.146   <- the level a model learns

  => a model that learns the all-day level, graded only on full-shelf days,
     looks 24% too high before it makes a single mistake

    sells_out  full_shelf_days  sold_out_days  demand_full_shelf  demand_all_days  looks_high_by_%
<25% censored            20009           4977              0.800            0.863              7.9
       25-50%           126583          77769              0.816            0.930             14.0
       50-75%            40671          57041              1.269            1.449             14.2
        >=75%             3030          17182              1.523            2.207             44.9


## 6. Where recovery pays, and where it costs

Section 4 is pooled, and section 5 explains why pooled comes back flat. Split the same forecasts by
**how often each product sells out** and the effect has somewhere to show up.

Notebook 01 ran this cut for the XGBoost baseline: the raw-target twin under-forecast the worst band
by 19.6%, and recovery removed essentially all of it for a 21.5% accuracy gain. The question is
whether the model we actually ship does the same.

**It can come back flat.** If it does, recovery helps a model we are not using, and the ordering stage
has nothing to convert into a saving. No refitting - the forecasts are already in memory.

In [10]:
tbl = pd.concat({tag: scores_by_bucket(fc["validation"], band)
                 for tag, fc in [("raw", raw), ("recovered", recovered)]}, axis=1)
tbl[("change", "WAPE_%")] = ((tbl[("recovered", "WAPE")] / tbl[("raw", "WAPE")] - 1) * 100).round(2)
print("negative WAPE_% = recovery is MORE accurate for that group of products")
print(tbl.round(4).to_string())

negative WAPE_% = recovery is MORE accurate for that group of products
                   raw                                              recovered                                              change
              n_scored    WAPE     WPE     MAE pinball(avg)   CRPS~  n_scored    WAPE     WPE     MAE pinball(avg)   CRPS~ WAPE_%
<25% censored   3955.0  0.4231 -0.0302  0.3066       0.0992  0.2722    3955.0  0.4207  0.0710  0.3047       0.0979  0.2684  -0.57
25-50%         30345.0  0.3512 -0.0445  0.2880       0.0931  0.2261   30345.0  0.3623  0.1052  0.2966       0.0944  0.2291   3.16
50-75%         12319.0  0.2982 -0.0670  0.4139       0.1333  0.1911   12319.0  0.2989  0.0835  0.4144       0.1331  0.1908   0.23
>=75%           1375.0  0.2595 -0.2017  0.7970       0.2655  0.1725    1375.0  0.1940 -0.0027  0.5877       0.1919  0.1247 -25.24
ALL            47994.0  0.3286 -0.0656  0.3364       0.1089  0.2120   47994.0  0.3284  0.0851  0.3358       0.1074  0.2091  -0.06


## 7. Overstock vs understock - the trade in units

WAPE charges the same for one unit too many as one unit too few. **A bin and an empty shelf are not
the same event**, and an empty shelf also produces another censored zero, which teaches the next model
to order even less. So the accuracy tables cannot express what recovery is for.

This splits the error by direction on days demand is known, ordering at `q50`:

- **understock %** - demand that walked away, as a share of demand. The failure recovery targets.
- **overstock %** - units that would have been binned. The price recovery charges.

Read the worst band first. If recovery is doing its job, understock falls sharply there and overstock
rises by less than it saves - and the cost sweep in the ordering stage decides whether that trade pays.

In [11]:
# One implementation, in src/utils/metrics.py - notebook 03 renders the same table for the tree,
# and a second copy here is how the two silently drift apart.
trade = lost_sales_vs_waste({"raw": raw["validation"], "recovered": recovered["validation"]}, band)
print(trade.to_string())
print()
print("worth_it_above_ratio = how much worse an empty shelf must be than a bin for recovery to pay.")
print("Below 1.0 means it pays even if waste and a lost sale cost exactly the same.")

               lost_sales_%_raw  waste_%_raw  lost_sales_%_recovered  waste_%_recovered  lost_sales_recovered_pts  waste_added_pts  worth_it_above_ratio
band                                                                                                                                                    
<25% censored              22.8         19.5                    17.6               24.4                       5.2              4.9                  0.94
25-50%                     19.8         15.4                    12.8               23.3                       7.0              7.9                  1.13
50-75%                     18.1         11.6                    10.8               19.0                       7.3              7.4                  1.01
>=75%                      22.7          2.8                     9.8                9.1                      12.9              6.3                  0.49

worth_it_above_ratio = how much worse an empty shelf must be than a bin for recov